<a href="https://colab.research.google.com/github/thomaslu678/Praxis-Lab-25-26/blob/main/clean/15_Automate_GIS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# INSTALLS (Colab)
# ============================================================
!pip install geopandas shapely pyogrio pandas numpy pyproj centerline==1.1.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 67.6 MB/s eta 0:00:00


In [3]:
# ============================================================
# IMPORTS
# ============================================================

import os
import json
import numpy as np
import pandas as pd

import geopandas as gpd

from shapely.geometry import (
    box,
    Point,
    MultiLineString
)

from centerline.geometry import Centerline

In [8]:
# ============================================================
# USER INPUTS
# ============================================================

case_study_path = "/content/Shape.gpkg"
parks_path = "/content/River.gpkg"
water_path = ""

output_dir = "/content/outputs"

In [6]:
os.makedirs(output_dir, exist_ok=True)

output_gpkg = os.path.join(
    output_dir,
    "workflow_outputs.gpkg"
)

metadata_json = os.path.join(
    output_dir,
    "metadata.json"
)

# remove previous gpkg if rerunning

if os.path.exists(output_gpkg):
    os.remove(output_gpkg)

In [7]:
# ============================================================
# HELPERS
# ============================================================

def save_layer(gdf, layer_name):

    gdf.to_file(
        output_gpkg,
        layer=layer_name,
        driver="GPKG",
        index=False
    )

    print(f"Saved layer: {layer_name}")


def union_geometry(gdf):
    """
    Version-safe dissolve.
    """

    try:
        return gdf.union_all()
    except Exception:
        return gdf.unary_union


In [9]:
# ============================================================
# LOAD INPUTS
# ============================================================

case_gdf = gpd.read_file(case_study_path)

parks_gdf = (
    gpd.read_file(parks_path)
    if parks_path
    else None
)

water_gdf = (
    gpd.read_file(water_path)
    if water_path
    else None
)

crs = case_gdf.crs

print("CRS:", crs)

CRS: EPSG:32617


In [10]:
# ============================================================
# STEP 1
# AREA + PERIMETER
# ============================================================

case_gdf = case_gdf.copy()

case_gdf["fid"] = range(len(case_gdf))

case_gdf["area"] = case_gdf.geometry.area
case_gdf["perimeter"] = case_gdf.geometry.length

save_layer(
    case_gdf,
    "00_case_metrics"
)

total_area = float(
    case_gdf["area"].sum()
)

total_perimeter = float(
    case_gdf["perimeter"].sum()
)

print("\nCASE STUDY METRICS")
print("------------------")
print("Area:", total_area)
print("Perimeter:", total_perimeter)

Saved layer: 00_case_metrics

CASE STUDY METRICS
------------------
Area: 44336.80760914131
Perimeter: 1612.607720928803


In [11]:
# ============================================================
# STEP 2
# CENTERLINE LENGTH
# ============================================================

case_polygon = case_gdf.geometry.iloc[0]

# repair geometry if needed

case_polygon = case_polygon.buffer(0)

centerline_length = None

try:

    centerline_obj = Centerline(
        case_polygon
    )

    centerline_geom = centerline_obj.geometry

    centerline_length = float(
        centerline_geom.length
    )

except Exception as e:

    print(
        "Centerline generation failed:",
        e
    )

    centerline_length = None

print(
    "Centerline Length:",
    centerline_length
)

Centerline Length: 928.1813599770699


In [12]:
# ============================================================
# STEP 3
# CASE BUFFER
# ============================================================

case_buffer = case_gdf.copy()

BUFFER_AMOUNT = 330

case_buffer["geometry"] = (
    case_buffer.geometry.buffer(BUFFER_AMOUNT)
)

save_layer(
    case_buffer,
    "01_case_buffer"
)

Saved layer: 01_case_buffer


In [13]:
# ============================================================
# STEP 4
# PARK BUFFER
# ============================================================

parks_buffer = None

if parks_gdf is not None:

    parks_buffer = parks_gdf.copy()

    parks_buffer["geometry"] = (
        parks_buffer.geometry.buffer(BUFFER_AMOUNT)
    )

    save_layer(
        parks_buffer,
        "02_park_buffer"
    )

Saved layer: 02_park_buffer


In [14]:
# ============================================================
# STEP 5
# CREATE GRID
# ============================================================

xmin, ymin, xmax, ymax = (
    case_buffer.total_bounds
)

cell_size = 30

polys = []
row_indices = []
col_indices = []

row_idx = 0

for y in np.arange(
    ymin,
    ymax,
    cell_size
):

    col_idx = 0

    for x in np.arange(
        xmin,
        xmax,
        cell_size
    ):

        polys.append(
            box(
                x,
                y,
                x + cell_size,
                y + cell_size
            )
        )

        row_indices.append(
            row_idx
        )

        col_indices.append(
            col_idx
        )

        col_idx += 1

    row_idx += 1

grid = gpd.GeoDataFrame(
    {
        "fid": range(len(polys)),
        "row_index": row_indices,
        "col_index": col_indices
    },
    geometry=polys,
    crs=crs
)

save_layer(
    grid,
    "03_grid"
)

current_grid = grid.copy()

Saved layer: 03_grid


In [15]:
# ============================================================
# STEP 6
# INTERSECT CASE BUFFER
# ============================================================

case_buffer_union = union_geometry(
    case_buffer
)

current_grid = current_grid[
    current_grid.intersects(
        case_buffer_union
    )
].copy()

save_layer(
    current_grid,
    "04_grid_case_buffer"
)


Saved layer: 04_grid_case_buffer


In [16]:
# ============================================================
# STEP 7
# DISJOINT PARK BUFFER
# ============================================================

if parks_buffer is not None:

    park_union = union_geometry(
        parks_buffer
    )

    current_grid = current_grid[
        ~current_grid.intersects(
            park_union
        )
    ].copy()

    save_layer(
        current_grid,
        "05_grid_no_parks"
    )

Saved layer: 05_grid_no_parks


In [18]:
# ============================================================
# STEP 8
# DISJOINT WATER
# ============================================================

if water_gdf is not None:

    water_union = union_geometry(
        water_gdf
    )

    current_grid = current_grid[
        ~current_grid.intersects(
            water_union
        )
    ].copy()

    save_layer(
        current_grid,
        "06_grid_no_water"
    )
else:
    print("No water layer")

No water layer


In [19]:
# ============================================================
# STEP 9
# CENTROIDS
# ============================================================

centroids = current_grid.copy()

centroids["geometry"] = (
    centroids.centroid
)

save_layer(
    centroids,
    "07_centroids"
)

Saved layer: 07_centroids


In [20]:
# ============================================================
# STEP 10
# DISTANCE TO CASE STUDY POLYGON
# ============================================================

centroids["distance"] = (
    centroids.geometry.distance(
        case_polygon
    )
)

save_layer(
    centroids,
    "08_centroids_distance"
)

Saved layer: 08_centroids_distance


In [21]:
# ============================================================
# STEP 11
# REPROJECT TO WGS84
# ============================================================

centroids_wgs84 = (
    centroids.to_crs(
        epsg=4326
    )
)

save_layer(
    centroids_wgs84,
    "09_centroids_wgs84"
)

Saved layer: 09_centroids_wgs84


In [22]:
# ============================================================
# STEP 12
# ADD XY FIELDS
# ============================================================

centroids_wgs84["x"] = (
    centroids_wgs84.geometry.x
)

centroids_wgs84["y"] = (
    centroids_wgs84.geometry.y
)

save_layer(
    centroids_wgs84,
    "10_centroids_xy"
)

Saved layer: 10_centroids_xy


In [23]:
# ============================================================
# STEP 13
# CREATE EXTENT LAYER
# ============================================================

xmin, ymin, xmax, ymax = (
    current_grid.total_bounds
)

extent_poly = box(
    xmin,
    ymin,
    xmax,
    ymax
)

extent_gdf = gpd.GeoDataFrame(
    {
        "fid": [0]
    },
    geometry=[extent_poly],
    crs=crs
)

save_layer(
    extent_gdf,
    "11_extent"
)


Saved layer: 11_extent


In [24]:
# ============================================================
# STEP 14
# BUFFER EXTENT BY 50m
# ============================================================

extent_buffer = extent_gdf.copy()

extent_buffer["geometry"] = (
    extent_buffer.geometry.buffer(50)
)

save_layer(
    extent_buffer,
    "12_extent_buffer"
)


Saved layer: 12_extent_buffer


In [25]:
# ============================================================
# STEP 15
# REPROJECT EXTENT TO WGS84
# ============================================================

extent_wgs84 = (
    extent_buffer.to_crs(
        epsg=4326
    )
)

save_layer(
    extent_wgs84,
    "13_extent_buffer_wgs84"
)

Saved layer: 13_extent_buffer_wgs84


In [26]:
# ============================================================
# STEP 16
# REPORT EXTENT
# ============================================================

xmin, ymin, xmax, ymax = (
    extent_wgs84.total_bounds
)

print("\nFINAL EXTENT (WGS84)")
print("--------------------")
print("xmin:", xmin)
print("ymin:", ymin)
print("xmax:", xmax)
print("ymax:", ymax)


FINAL EXTENT (WGS84)
--------------------
xmin: -80.00780346066563
ymin: 40.43949159525424
xmax: -79.99383157343098
ymax: 40.44455591358872


In [27]:
# ============================================================
# STEP 17
# METADATA TABLE
# ============================================================

metadata_df = pd.DataFrame(
    {
        "key": [
            "total_area",
            "total_perimeter",
            "centerline_length",
            "candidate_count",
            "xmin",
            "ymin",
            "xmax",
            "ymax"
        ],
        "value": [
            total_area,
            total_perimeter,
            centerline_length,
            len(centroids_wgs84),
            xmin,
            ymin,
            xmax,
            ymax
        ]
    }
)

metadata_gdf = gpd.GeoDataFrame(
    metadata_df,
    geometry=[None] * len(metadata_df),
    crs=crs
)

save_layer(
    metadata_gdf,
    "metadata"
)

Saved layer: metadata


In [28]:
# ============================================================
# STEP 18
# METADATA JSON
# ============================================================

metadata = {

    "total_area":
        total_area,

    "total_perimeter":
        total_perimeter,

    "centerline_length":
        centerline_length,

    "candidate_count":
        int(
            len(
                centroids_wgs84
            )
        ),

    "extent_wgs84": {

        "xmin":
            float(xmin),

        "ymin":
            float(ymin),

        "xmax":
            float(xmax),

        "ymax":
            float(ymax)
    }
}

with open(
    metadata_json,
    "w"
) as f:

    json.dump(
        metadata,
        f,
        indent=2
    )

print("\nMetadata saved:")
print(metadata_json)

print("\nGeoPackage saved:")
print(output_gpkg)

print("\nWorkflow complete.")


Metadata saved:
/content/outputs/metadata.json

GeoPackage saved:
/content/outputs/workflow_outputs.gpkg

Workflow complete.
